In [8]:
import polars as pl

file_path = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\raw\complaints.csv"

df = pl.read_csv(file_path, infer_schema_length=10000, ignore_errors=True)

# Products we want for the project
target_products = [
    "Credit card",
    "Credit card or prepaid card",
    "Checking or savings account",
    "Bank account or service",
    "Mortgage",
    "Debt collection",
    "Money transfer, virtual currency, or money service",
    "Money transfers",
    "Credit reporting, credit repair services, or other personal consumer reports",
    "Credit reporting"
]

# Select useful columns
df = df.select([
    "Date received",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative",
    "Company",
    "State",
    "Submitted via",
    "Timely response?",
    "Consumer disputed?",
    "Complaint ID"
])

# Rename columns
df = df.rename({
    "Date received": "date_received",
    "Product": "product",
    "Sub-product": "sub_product",
    "Issue": "issue",
    "Sub-issue": "sub_issue",
    "Consumer complaint narrative": "complaint_text",
    "Company": "company",
    "State": "state",
    "Submitted via": "submitted_via",
    "Timely response?": "timely_response",
    "Consumer disputed?": "consumer_disputed",
    "Complaint ID": "complaint_id"
})

# Convert date
df = df.with_columns(
    pl.col("date_received").str.strptime(pl.Date, strict=False)
)

# Filter recent complaints
df = df.filter(
    (pl.col("date_received").dt.year() >= 2021) &
    (pl.col("product").is_in(target_products))
)

# Remove duplicates
df = df.unique(subset=["complaint_id"])

# Create NLP dataset (rows with text)
text_df = df.filter(
    pl.col("complaint_text").is_not_null()
)

# Reduce size if too large
if df.height > 100000:
    df = df.sample(n=100000, seed=42)

if text_df.height > 30000:
    text_df = text_df.sample(n=30000, seed=42)

# Save files
output = Path(r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed")

df.write_csv(output / "complaints_clean.csv")
text_df.write_csv(output / "complaints_text.csv")

In [7]:
from pathlib import Path

input_file = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed"

print("File exists:", Path(input_file).exists())
print("Path:", input_file)

File exists: True
Path: C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed
